# Hindi / Hinglish intent classifier: train + UI on Colab (T4)

Run the cells **in order**. Each step says what to look for before moving on.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU** > Save.
2. In your Google Drive, create a folder called `nlu` and put these 6 files in it:
   `nlu_pipeline.py`, `nlu_infer.py`, `train_final.py`, `app.py`, `train.csv`, `test.csv`

## Step 1: Check the GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then re-run.")

CUDA available: True
Tesla T4


## Step 2: Mount Drive and check your files
Allow access when the pop-up asks. Change `PROJECT` if your folder has a different name.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = "/content/drive/MyDrive/nlu"
need = ["nlu_pipeline.py", "nlu_infer.py", "train_final.py", "app.py", "train.csv", "test.csv"]
missing = [f for f in need if not os.path.exists(os.path.join(PROJECT, f))]
print("Missing files:", missing if missing else "none, all good")
assert not missing, "Upload the missing files to the Drive folder first."
%cd $PROJECT

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving nlu_pipeline.py to nlu_pipeline.py
Saving nlu_infer.py to nlu_infer.py
Saving app.py to app.py
Saving train_final.py to train_final.py
Saving train.csv to train.csv
Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv


In [ ]:
import os

need = [
    "nlu_pipeline.py",
    "nlu_infer.py",
    "train_final.py",
    "app.py",
    "train.csv",
    "test.csv"
]

missing = [f for f in need if not os.path.exists(f)]

print("Missing files:", missing if missing else "All files uploaded!")

Missing files: All files uploaded!


## Step 3: Install the extra packages
(torch, transformers, scikit-learn and pandas are already on Colab.) Takes about a minute.

In [ ]:
!pip install -q -U "gradio>=6" indic-transliteration

## Step 4: Quick smoke test (about 3-5 minutes)
Trains on 3,000 rows for 1 epoch. This downloads MuRIL (~950 MB) and proves the whole pipeline works on your GPU **before** you spend 30+ minutes on the real run. The score here will be low; that is expected.

Look for: `Transformer (holdout)`, `parameters: ... (limit 300M)`, `saved bundle reloaded`, `wrote ... submission_smoke.csv`.
If MuRIL cannot be downloaded, add `--model xlm-roberta-base` to this command and to the Step 5 command.

In [ ]:
!python train_final.py --data_dir . --out_dir /content/smoke --smoke --epochs 1

## Step 5: Full training (roughly 20-40 minutes, estimated)
Trains on 90% of the data, tunes the blend on the 10% holdout, then retrains on 100% (`--full_refit`). Output goes to Drive, so the model survives a disconnect.

**What good looks like in the log**
- `SVM (holdout)` should be about 0.83.
- `Transformer (holdout)` should be at or above the SVM. If it is clearly below, try `--lr 2e-5` or `--epochs 6`.
- `blend weight on transformer` shows how much the ensemble leans on MuRIL.
- `parameters: ... = ...M (limit 300M)` must be under 300M (expect about 244M).
- The final `saved bundle reloaded: holdout macro-F1` is optimistic with `--full_refit` (holdout rows were in training). Use the earlier `holdout` lines as your honest estimate.

Keep this browser tab open and the session connected while it runs.
If you run out of GPU memory, add `--bs 16`.

In [ ]:
!python train_final.py --data_dir . --out_dir "$PROJECT/final" --full_refit

[09:08:03] groups: sim threshold 0.80 -> 12196 groups, largest 66 rows, 3044 multi-row groups
[09:08:13] train 15237 rows / holdout 1699 rows
[09:08:13] fitting SVM
[09:08:24] SVM (holdout): macro-F1 0.8197 | Devanagari 0.8150 | romanized 0.8001
[09:08:24]    worst classes: 37:0.56, 14:0.59, 28:0.60, 12:0.63, 17:0.63, 27:0.64, 34:0.67, 53:0.68
config.json: 100% 411/411 [00:00<00:00, 1.80MB/s]
tokenizer_config.json: 100% 206/206 [00:00<00:00, 939kB/s]
vocab.txt: 100% 3.16M/3.16M [00:00<00:00, 114MB/s]
special_tokens_map.json: 100% 113/113 [00:00<00:00, 528kB/s]
[09:08:33] fine-tuning transformer: google/muril-base-cased

pytorch_model.bin: downloading bytes:  33% 319M/953M [00:03<00:04, 140MB/s, 26.5MB/s  ]
pytorch_model.bin: reconstructing file:  23% 219M/953M [00:03<00:12, 58.2MB/s]
pytorch_model.bin: downloading bytes:  36% 343M/953M [00:03<00:04, 130MB/s, 28.2MB/s  ]
pytorch_model.bin: downloading bytes:  47% 451M/953M [00:04<00:02, 195MB/s, 35.5MB/s  ]
pytorch_model.bin: downloadin

## Step 6: Check and download the submission file

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if "submission" in file.lower() or file.endswith(".csv"):
            print(os.path.join(root, file))

/content/test.csv
/content/train.csv
/content/sample_submission.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_train.csv


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'nlu_pipeline.py', 'test.csv', '__pycache__', 'nlu_infer.py', 'train.csv', 'sample_submission.csv', 'app.py', 'train_final.py', 'sample_data']


In [ ]:
%cd /content
!python train_final.py

/content
[10:01:11] groups: sim threshold 0.80 -> 12196 groups, largest 66 rows, 3044 multi-row groups
[10:01:20] train 15237 rows / holdout 1699 rows
[10:01:20] fitting SVM
[10:01:32] SVM (holdout): macro-F1 0.8197 | Devanagari 0.8150 | romanized 0.8001
[10:01:32]    worst classes: 37:0.56, 14:0.59, 28:0.60, 12:0.63, 17:0.63, 27:0.64, 34:0.67, 53:0.68
[10:01:37] fine-tuning transformer: google/muril-base-cased
Loading weights: 100% 199/199 [00:00<00:00, 20686.69it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.wei

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if "submission" in file.lower():
            print(os.path.join(root, file))

/content/sample_submission.csv
/content/final/submission.csv


In [ ]:
import pandas as pd

sub = pd.read_csv("/content/final/submission.csv")
test = pd.read_csv("/content/test.csv")

print("Submission shape:", sub.shape)
print("Columns:", sub.columns.tolist())

assert len(sub) == len(test), "Row count mismatch!"
assert (sub.example_id.values == test.example_id.values).all(), "ID order mismatch!"
assert sub.target.between(0, 55).all(), "Invalid class labels!"

print("Submission format is valid!")
print("Class 55 share: %.1f%%" % (100 * (sub.target == 55).mean()))

display(sub.head())

Submission shape: (4912, 2)
Columns: ['example_id', 'target']
Submission format is valid!
Class 55 share: 7.2%


,example_id,target
0,900001,43
1,900002,38
2,900003,43
3,900004,44
4,900005,48


In [ ]:
from google.colab import files

files.download("/content/final/submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 7: Launch the UI
The cell keeps running while the app is up. Look for the line **`Running on public URL: https://....gradio.live`** and open it in a new tab.
The link works only while this cell runs (and for up to one week). Stop the cell to shut the UI down.

In [ ]:
!python app.py --bundle "$PROJECT/final/bundle" --share

Loading weights: 100% 201/201 [00:00<00:00, 4867.33it/s]
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://27e7ed0e197c0dafc2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 3959, in block_thread
    time.sleep(0.1)
    ~~~~~~~~~~^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/app.py", line 110, in <module>
    main()
    ~~~~^^
  File "/content/app.py", line 106, in main
    build_ui(clf).launch(server_name=args.host, server_port=args.port, share=args.share, **launch_kw)
    ~~~~~~~~~~~~~~~

## Optional: use the model from Python (no UI)
Run this in a new cell after Step 5 (stop the Step 7 cell first, or run it in a separate session).

In [ ]:
import sys; sys.path.insert(0, PROJECT)
from nlu_infer import IntentClassifier
clf = IntentClassifier(f"{PROJECT}/final/bundle")
for r in clf.predict(["kripaya rasoeeghar ki battiyaan band kar den", "आज मौसम कैसा है"], topk=3):
    print(r["text"], "->", clf.name(r["pred"]), f"{r['confidence']:.1%}", "| top3:", r["top"])

## Troubleshooting
| Problem | Fix |
|---|---|
| `ModuleNotFoundError: nlu_pipeline` | Re-run Step 2 so `%cd` puts you in the project folder. |
| CUDA out of memory | Add `--bs 16` to the train command. |
| MuRIL download fails | Add `--model xlm-roberta-base` (about 278M params, still under the limit). |
| Session disconnected mid-training | Reconnect, re-run Steps 1-3, then Step 5 again (training restarts; files already on Drive are kept). |
| No `gradio.live` link | Re-run Step 3, then Step 7. |